# HW03 — Model Serving & Deployment

Your model is trained and tracked in MLflow. A model that only runs in a notebook is not useful in production.

In this homework you will:

- verify that your trained model is reproducible and ready for serving
- wrap it in a FastAPI service with proper input validation and batch support
- measure the performance difference between single and batch prediction
- package the service in Docker with a size-optimized image
- write Kubernetes manifests to deploy the service

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords, tokens, or notebook checkpoints.
Do not hardcode passwords anywhere in your code.

## Useful references

- MLflow model loading: https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html
- FastAPI: https://fastapi.tiangolo.com/
- FastAPI lifespan: https://fastapi.tiangolo.com/advanced/events/
- Pydantic v2: https://docs.pydantic.dev/latest/
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker multi-stage builds: https://docs.docker.com/build/building/multi-stage/
- Kubernetes Deployments: https://kubernetes.io/docs/concepts/workloads/controllers/deployment/
- Kubernetes Services: https://kubernetes.io/docs/concepts/services-networking/service/

## What to avoid

- Loading the model inside the request handler. Load once at startup.
- Hardcoded passwords in any source file.
- A Docker image that bakes in the model file. Pull from MLflow at startup.
- Returning raw numpy types from the API. JSON needs native Python types.
- Skipping the batch vs single benchmark. The numbers tell a story.

In [18]:
import os
import re
import time
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import joblib
from mlflow.tracking import MlflowClient

PROJECT = Path.cwd()
for path in ['src/airbnb_serving', 'k8s', 'reports', 'screenshots']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_serving/__init__.py').write_text('__version__ = "0.1.0"\n')

STUDENT_ID = os.getenv('QBC12_STUDENT_ID', 'student_ali_ghanbari').strip()
safe_student = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
EXPERIMENT_NAME = f'qbc12_hw02_{safe_student}'

print('PROJECT:', PROJECT)
print('EXPERIMENT_NAME:', EXPERIMENT_NAME)


PROJECT: /Users/mahbod/Desktop/Quera/Mlops/deadlines/W3
EXPERIMENT_NAME: qbc12_hw02_student_ali_ghanbari


---
## Part 1 — Model Reproducibility Check

Before serving a model, you must verify it produces exactly the same output as it did during training.

This is called a **reproducibility check** and it catches silent bugs like:
- preprocessing mismatch between training and serving
- wrong model version loaded
- feature column order changed

### 1.1 Connect to MLflow and load your best model

In [2]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://185.50.38.163:33014')
MLFLOW_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME', '') or input('MLflow username: ').strip()
MLFLOW_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD', '') or input('MLflow password: ').strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f'Experiment not found: {EXPERIMENT_NAME}. Complete HW02 first.')

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.leakage_status = 'clean' and tags.selected_for_serving = 'true'",
    order_by=['metrics.f1 DESC'],
)

if runs.empty:
    raise ValueError(
        'No run tagged selected_for_serving=true found. '
        'Go to MLflow UI, find your best clean run, and add the tag.'
    )

BEST_RUN_ID = runs.iloc[0]['run_id']
MODEL_URI = f'runs:/{BEST_RUN_ID}/model'

def load_serving_model(model_uri: str):
    local_model = Path(os.getenv('LOCAL_MODEL_PATH', PROJECT / 'artifacts/hw02_model/pipeline.joblib'))
    if local_model.exists():
        print('Loading cached joblib model:', local_model)
        return joblib.load(local_model)
    try:
        return mlflow.sklearn.load_model(model_uri)
    except Exception as exc:
        print('MLflow sklearn flavor load failed; trying joblib fallback:', exc)
        artifact_path = Path(mlflow.artifacts.download_artifacts(artifact_uri=model_uri))
        joblib_candidates = list(artifact_path.rglob('*.joblib'))
        if not joblib_candidates:
            raise
        print('Loading joblib artifact:', joblib_candidates[0])
        return joblib.load(joblib_candidates[0])
print('Best run ID  :', BEST_RUN_ID)
print('Model URI    :', MODEL_URI)
print('Run name     :', runs.iloc[0].get('tags.mlflow.runName'))
print('F1 score     :', runs.iloc[0].get('metrics.f1'))


Best run ID  : 84f84974651a401d943de5505b8202b0
Model URI    : runs:/84f84974651a401d943de5505b8202b0/model
Run name     : v5_random_forest
F1 score     : 0.9075380616412922


In [3]:
model = load_serving_model(MODEL_URI)
print('Model type:', type(model))
print('Model steps:', list(model.named_steps.keys()) if hasattr(model, 'named_steps') else 'not a pipeline')


Loading cached joblib model: /Users/mahbod/Desktop/Quera/Mlops/deadlines/W3/artifacts/hw02_model/pipeline.joblib


Model type: <class 'sklearn.pipeline.Pipeline'>
Model steps: ['preprocessor', 'model']


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying

### 1.2 Load your HW01 feature dataset

You will use a small sample from your HW01 feature parquet file to verify reproducibility.

In [4]:
FEATURE_COLS = [
    'room_type', 'property_type', 'neighbourhood_name',
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price',
    'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost',
    'host_listing_count', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff',
    'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff',
    'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d',
    'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d',
    'available_days_last_30d', 'available_rate_last_30d',
    'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d',
]
TARGET_COL = 'high_demand_proxy'

# Load your HW01/W2 parquet file
# Adjust the path if needed. This notebook lives in W3, so we also look in ../W2.
feature_candidates = [
    *Path('data/features').glob('*.parquet'),
    *Path('../W2/data/features').glob('*.parquet'),
    *Path('../W1/data/features').glob('*.parquet'),
]
if not feature_candidates:
    raise FileNotFoundError('Feature parquet not found. Checked data/features, ../W2/data/features, and ../W1/data/features.')

parquet_path = sorted(feature_candidates)[0]
df = pd.read_parquet(parquet_path)
print('Loaded feature parquet:', parquet_path)
if 'is_superhost' not in df.columns and 'host_is_superhost' in df.columns:
    df['is_superhost'] = df['host_is_superhost']
print('Dataset shape:', df.shape)
df[FEATURE_COLS + [TARGET_COL]].head(3)


Loaded feature parquet: ../W2/data/features/listing_availability_features_v1_audit_cleaned.parquet
Dataset shape: (10480, 34)


,room_type,property_type,neighbourhood_name,accommodates,bedrooms,beds,bathrooms,listing_price,minimum_nights,maximum_nights,...,days_since_last_review,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,high_demand_proxy
0,Private room,Private room in houseboat,Centrum-West,2,1.0,1.0,1.5,132.0,3,356,...,338.0,0,0.000000,3.0,30.0,0,0.000000,3.0,30.0,1
1,Private room,Private room in rental unit,Centrum-West,2,1.0,1.0,1.0,89.0,2,730,...,338.0,45,0.500000,2.0,730.0,14,0.466667,2.0,730.0,0
2,Private room,Private room in condo,Centrum-Oost,2,1.0,1.0,1.0,61.0,2,730,...,337.0,42,0.466667,2.0,730.0,16,0.533333,2.0,730.0,1


### 1.3 Reproducibility check

**TODO 1.3**

Take a sample of 50 rows from the dataset.

Run `model.predict()` on those rows **twice** and verify the results are identical.

Then compare the predictions against the `high_demand_proxy` column and print:
- how many predictions match the training label
- the accuracy on this sample

If both runs produce identical output, print `Reproducibility check passed.`
If they differ, raise a `ValueError`.

In [5]:

# TODO 1.3
# Write your reproducibility check here.

model = load_serving_model(MODEL_URI)

sample = df[FEATURE_COLS + [TARGET_COL]].sample(50, random_state=42)
X_sample = sample[FEATURE_COLS].astype(object).where(pd.notna(sample[FEATURE_COLS]), np.nan)
y_sample = sample[TARGET_COL]

pred_1 = model.predict(X_sample)
pred_2 = model.predict(X_sample)

if not np.array_equal(pred_1, pred_2):
    raise ValueError('Reproducibility check failed: two prediction runs differ.')

matches = int((pred_1 == y_sample.to_numpy()).sum())
accuracy = matches / len(y_sample)

print(f'Matching predictions: {matches}/{len(y_sample)}')
print(f'Sample accuracy: {accuracy:.4f}')
print('Reproducibility check passed.')


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying

Loading cached joblib model: /Users/mahbod/Desktop/Quera/Mlops/deadlines/W3/artifacts/hw02_model/pipeline.joblib
Matching predictions: 47/50
Sample accuracy: 0.9400
Reproducibility check passed.


---
## Part 2 — FastAPI Service

A REST API is the standard way to expose an ML model to other systems.

You will build a FastAPI app with two prediction endpoints:
- `POST /predict` — single listing prediction
- `POST /predict/batch` — multiple listings in one request

Then you will measure how much faster batch is compared to calling single predict in a loop.

### 2.1 Input and output schemas

In [6]:

# TODO 2.1
# Create src/airbnb_serving/schema.py
#
# Define two Pydantic models:
#
# ListingFeatures:
#   - all feature columns from FEATURE_COLS above
#   - use correct types: str, int, float, bool
#   - nullable fields (those with NaN in dataset) should use Optional[float] = None
#
# PredictionResponse:
#   - listing_id: int | None = None
#   - prediction: int  (0 or 1)
#   - probability_high_demand: float
#   - model_run_id: str
#
# Write your code here.

(PROJECT / 'src/airbnb_serving/schema.py').write_text(textwrap.dedent('''
from __future__ import annotations

from typing import Optional

from pydantic import BaseModel, ConfigDict, Field


class ListingFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")

    room_type: str
    property_type: str
    neighbourhood_name: str
    accommodates: int
    bedrooms: Optional[float] = None
    beds: Optional[float] = None
    bathrooms: Optional[float] = None
    listing_price: float
    minimum_nights: int
    maximum_nights: int
    instant_bookable: bool
    is_superhost: bool
    host_listing_count: float
    total_reviews_before_cutoff: float
    unique_reviewers_before_cutoff: float
    avg_comment_len_before_cutoff: Optional[float] = None
    max_comment_len_before_cutoff: Optional[float] = None
    days_since_last_review: Optional[float] = None
    available_days_last_90d: float
    available_rate_last_90d: float
    avg_minimum_nights_calendar_last_90d: Optional[float] = None
    avg_maximum_nights_calendar_last_90d: Optional[float] = None
    available_days_last_30d: float
    available_rate_last_30d: float
    avg_minimum_nights_calendar_last_30d: Optional[float] = None
    avg_maximum_nights_calendar_last_30d: Optional[float] = None


class PredictionResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")

    listing_id: int | None = None
    prediction: int = Field(..., ge=0, le=1)
    probability_high_demand: float = Field(..., ge=0.0, le=1.0)
    model_run_id: str
''').strip() + '\n')

print((PROJECT / 'src/airbnb_serving/schema.py').read_text())


from __future__ import annotations

from typing import Optional

from pydantic import BaseModel, ConfigDict, Field


class ListingFeatures(BaseModel):
    model_config = ConfigDict(extra="forbid")

    room_type: str
    property_type: str
    neighbourhood_name: str
    accommodates: int
    bedrooms: Optional[float] = None
    beds: Optional[float] = None
    bathrooms: Optional[float] = None
    listing_price: float
    minimum_nights: int
    maximum_nights: int
    instant_bookable: bool
    is_superhost: bool
    host_listing_count: float
    total_reviews_before_cutoff: float
    unique_reviewers_before_cutoff: float
    avg_comment_len_before_cutoff: Optional[float] = None
    max_comment_len_before_cutoff: Optional[float] = None
    days_since_last_review: Optional[float] = None
    available_days_last_90d: float
    available_rate_last_90d: float
    avg_minimum_nights_calendar_last_90d: Optional[float] = None
    avg_maximum_nights_calendar_last_90d: Optional[float] = None
 

### 2.2 Prediction logic

In [7]:

# TODO 2.2
# Create src/airbnb_serving/predictor.py
#
# Add two functions:
#
# predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse
#   - convert ListingFeatures to a single-row DataFrame
#   - call model.predict and model.predict_proba
#   - return PredictionResponse
#   - all values must be native Python types (int, float), not numpy types
#
# predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]
#   - convert list to a multi-row DataFrame in one step
#   - call model.predict and model.predict_proba once for the whole batch
#   - return a list of PredictionResponse
#   - do NOT loop and call predict_single for each row

(PROJECT / 'src/airbnb_serving/predictor.py').write_text(textwrap.dedent('''
from __future__ import annotations

import numpy as np
import pandas as pd

from .schema import ListingFeatures, PredictionResponse


def _sanitize_frame(X: pd.DataFrame) -> pd.DataFrame:
    return X.astype(object).where(pd.notna(X), np.nan)


def _probability_high_demand(model, X: pd.DataFrame, predictions) -> list[float]:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] > 1:
            return [float(x) for x in proba[:, 1]]
        return [float(x) for x in proba.reshape(-1)]
    return [float(x) for x in predictions]


def predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse:
    X = _sanitize_frame(pd.DataFrame([features.model_dump()]))
    predictions = model.predict(X)
    probs = _probability_high_demand(model, X, predictions)
    return PredictionResponse(
        prediction=int(predictions[0]),
        probability_high_demand=float(probs[0]),
        model_run_id=run_id,
    )


def predict_batch(features_list: list[ListingFeatures], model, run_id: str) -> list[PredictionResponse]:
    X = _sanitize_frame(pd.DataFrame([features.model_dump() for features in features_list]))
    predictions = model.predict(X)
    probs = _probability_high_demand(model, X, predictions)
    return [
        PredictionResponse(
            prediction=int(pred),
            probability_high_demand=float(prob),
            model_run_id=run_id,
        )
        for pred, prob in zip(predictions, probs)
    ]
''').strip() + '\n')

print((PROJECT / 'src/airbnb_serving/predictor.py').read_text())


from __future__ import annotations

import numpy as np
import pandas as pd

from .schema import ListingFeatures, PredictionResponse


def _sanitize_frame(X: pd.DataFrame) -> pd.DataFrame:
    return X.astype(object).where(pd.notna(X), np.nan)


def _probability_high_demand(model, X: pd.DataFrame, predictions) -> list[float]:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] > 1:
            return [float(x) for x in proba[:, 1]]
        return [float(x) for x in proba.reshape(-1)]
    return [float(x) for x in predictions]


def predict_single(features: ListingFeatures, model, run_id: str) -> PredictionResponse:
    X = _sanitize_frame(pd.DataFrame([features.model_dump()]))
    predictions = model.predict(X)
    probs = _probability_high_demand(model, X, predictions)
    return PredictionResponse(
        prediction=int(predictions[0]),
        probability_high_demand=float(probs[0]),
        model_run_id=run_id

### 2.3 FastAPI app

In [8]:

# TODO 2.3
# Create src/airbnb_serving/app.py
#
# Build a FastAPI app with:
#
#   GET /health
#     response: {"status": "ok", "model_run_id": str}
#
#   POST /predict
#     request body: ListingFeatures
#     response: PredictionResponse
#
#   POST /predict/batch
#     request body: list[ListingFeatures]
#     response: list[PredictionResponse]
#
# Rules:
#   - Load the model ONCE using a lifespan context manager, not inside handlers
#   - Read MODEL_RUN_ID, MLFLOW_TRACKING_URI, MLFLOW_TRACKING_USERNAME,
#     MLFLOW_TRACKING_PASSWORD from environment variables
#   - Use the predictor module for prediction logic

(PROJECT / 'src/airbnb_serving/app.py').write_text(textwrap.dedent('''
from __future__ import annotations

import os
from contextlib import asynccontextmanager

import mlflow
import mlflow.sklearn
import joblib
from pathlib import Path
from fastapi import FastAPI, HTTPException

from .predictor import predict_batch, predict_single
from .schema import ListingFeatures, PredictionResponse

_model = None
_model_run_id = ""


def load_serving_model(model_uri: str):
    local_model = Path(os.getenv("LOCAL_MODEL_PATH", "artifacts/hw02_model/pipeline.joblib"))
    if local_model.exists():
        return joblib.load(local_model)
    try:
        return mlflow.sklearn.load_model(model_uri)
    except Exception:
        artifact_path = Path(mlflow.artifacts.download_artifacts(artifact_uri=model_uri))
        joblib_candidates = list(artifact_path.rglob("*.joblib"))
        if not joblib_candidates:
            raise
        return joblib.load(joblib_candidates[0])


@asynccontextmanager
async def lifespan(app: FastAPI):
    global _model, _model_run_id
    _model_run_id = os.environ["MODEL_RUN_ID"]
    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://185.50.38.163:33014"))
    if os.getenv("MLFLOW_TRACKING_USERNAME"):
        os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "")
    if os.getenv("MLFLOW_TRACKING_PASSWORD"):
        os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "")
    _model = load_serving_model(f"runs:/{_model_run_id}/model")
    yield
    _model = None


app = FastAPI(title="QBC12 Airbnb Serving", version="0.1.0", lifespan=lifespan)


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok", "model_run_id": _model_run_id}


@app.post("/predict", response_model=PredictionResponse)
def predict(features: ListingFeatures) -> PredictionResponse:
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")
    return predict_single(features, _model, _model_run_id)


@app.post("/predict/batch", response_model=list[PredictionResponse])
def batch(features: list[ListingFeatures]) -> list[PredictionResponse]:
    if _model is None:
        raise HTTPException(status_code=503, detail="model not loaded")
    if not features:
        raise HTTPException(status_code=422, detail="batch must not be empty")
    return predict_batch(features, _model, _model_run_id)
''').strip() + '\n')

print((PROJECT / 'src/airbnb_serving/app.py').read_text())


from __future__ import annotations

import os
from contextlib import asynccontextmanager

import mlflow
import mlflow.sklearn
import joblib
from pathlib import Path
from fastapi import FastAPI, HTTPException

from .predictor import predict_batch, predict_single
from .schema import ListingFeatures, PredictionResponse

_model = None
_model_run_id = ""


def load_serving_model(model_uri: str):
    local_model = Path(os.getenv("LOCAL_MODEL_PATH", "artifacts/hw02_model/pipeline.joblib"))
    if local_model.exists():
        return joblib.load(local_model)
    try:
        return mlflow.sklearn.load_model(model_uri)
    except Exception:
        artifact_path = Path(mlflow.artifacts.download_artifacts(artifact_uri=model_uri))
        joblib_candidates = list(artifact_path.rglob("*.joblib"))
        if not joblib_candidates:
            raise
        return joblib.load(joblib_candidates[0])


@asynccontextmanager
async def lifespan(app: FastAPI):
    global _model, _model_run_id
    _model_ru

### 2.4 Package metadata

In [9]:

# TODO 2.4
# Create pyproject.toml and requirements.txt
#
# Package name: airbnb-serving
# Source directory: src/
# Required dependencies:
#   fastapi>=0.111
#   uvicorn[standard]>=0.29
#   mlflow>=2.13
#   scikit-learn>=1.4
#   pandas>=2.0
#   pydantic>=2.0
#   python-dotenv>=1.0

# Write your code here.

requirements = [
    'fastapi>=0.111',
    'uvicorn[standard]>=0.29',
    'mlflow>=2.13',
    'scikit-learn>=1.4',
    'pandas>=2.0',
    'pydantic>=2.0',
    'python-dotenv>=1.0',
    'joblib>=1.3',
]
Path('requirements.txt').write_text('\n'.join(requirements) + '\n')

Path('pyproject.toml').write_text(textwrap.dedent('''
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-serving"
version = "0.1.0"
description = "QBC12 HW03 Airbnb model serving API"
requires-python = ">=3.11"
dependencies = [
  "fastapi>=0.111",
  "uvicorn[standard]>=0.29",
  "mlflow>=2.13",
  "scikit-learn>=1.4",
  "pandas>=2.0",
  "pydantic>=2.0",
  "python-dotenv>=1.0",
  "joblib>=1.3",
]

[tool.setuptools.packages.find]
where = ["src"]
''').strip() + '\n')

print(Path('pyproject.toml').read_text())


[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-serving"
version = "0.1.0"
description = "QBC12 HW03 Airbnb model serving API"
requires-python = ">=3.11"
dependencies = [
  "fastapi>=0.111",
  "uvicorn[standard]>=0.29",
  "mlflow>=2.13",
  "scikit-learn>=1.4",
  "pandas>=2.0",
  "pydantic>=2.0",
  "python-dotenv>=1.0",
  "joblib>=1.3",
]

[tool.setuptools.packages.find]
where = ["src"]



### 2.5 Local install and smoke test

Install the package and manually start the server in a terminal before running the test cell below.

```bash
pip install -e .

MODEL_RUN_ID=<your_run_id> \
MLFLOW_TRACKING_URI=http://185.50.38.163:33014 \
MLFLOW_TRACKING_USERNAME=<user> \
MLFLOW_TRACKING_PASSWORD=<pass> \
uvicorn airbnb_serving.app:app --host 0.0.0.0 --port 8000
```

In [10]:
import sys
!{sys.executable} -m pip install -q -e .


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip3.11 install --upgrade pip


In [11]:
import subprocess, time, signal, os

server_proc = subprocess.Popen(
    ['uvicorn', 'airbnb_serving.app:app', '--host', '0.0.0.0', '--port', '12345'],
    env={**os.environ, 'MODEL_RUN_ID': BEST_RUN_ID, 'SERVING_URL': 'http://127.0.0.1:12345', 'LOCAL_MODEL_PATH': str(PROJECT / 'artifacts/hw02_model/pipeline.joblib')},
)
time.sleep(10)  # wait for cached model to load
print('Server started, PID:', server_proc.pid)


INFO:     Started server process [70144]
INFO:     Waiting for application startup.


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying

Server started, PID: 70144


In [12]:
import requests

BASE_URL = 'http://localhost:12345'

health = requests.get(f'{BASE_URL}/health')
assert health.status_code == 200, f'Health check failed: {health.text}'
print('Health:', health.json())

sample_payload = {
    'room_type': 'Entire home/apt',
    'property_type': 'Entire rental unit',
    'neighbourhood_name': 'Centrum-West',
    'accommodates': 2,
    'bedrooms': 1.0,
    'beds': 1.0,
    'bathrooms': 1.0,
    'listing_price': 150.0,
    'minimum_nights': 2,
    'maximum_nights': 365,
    'instant_bookable': True,
    'is_superhost': False,
    'host_listing_count': 1,
    'total_reviews_before_cutoff': 10.0,
    'unique_reviewers_before_cutoff': 9.0,
    'avg_comment_len_before_cutoff': 120.0,
    'max_comment_len_before_cutoff': 300.0,
    'days_since_last_review': 30.0,
    'available_days_last_90d': 45,
    'available_rate_last_90d': 0.5,
    'avg_minimum_nights_calendar_last_90d': 2.0,
    'avg_maximum_nights_calendar_last_90d': 365.0,
    'available_days_last_30d': 15,
    'available_rate_last_30d': 0.5,
    'avg_minimum_nights_calendar_last_30d': 2.0,
    'avg_maximum_nights_calendar_last_30d': 365.0,
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_payload)
assert resp.status_code == 200, f'Single predict failed: {resp.text}'
print('Single predict:', resp.json())

print('Local smoke test passed.')


INFO:     127.0.0.1:49173 - "GET /health HTTP/1.1" 200 OK
Health: {'status': 'ok', 'model_run_id': '84f84974651a401d943de5505b8202b0'}
INFO:     127.0.0.1:49175 - "POST /predict HTTP/1.1" 200 OK
Single predict: {'listing_id': None, 'prediction': 0, 'probability_high_demand': 0.22348260870782685, 'model_run_id': '84f84974651a401d943de5505b8202b0'}
Local smoke test passed.


### 2.6 Batch vs single benchmark

**TODO 2.6**

Take 100 rows from your feature dataset.

Measure:
1. Time to call `POST /predict` 100 times in a loop (single)
2. Time to call `POST /predict/batch` once with all 100 rows (batch)

Print a comparison table with total time and time per prediction for each approach.

Then answer: why is batch faster? Write your answer as a comment in the cell.

In [13]:

# TODO 2.6
# Write your benchmark here.
# Hint: convert df rows to list of dicts with df.to_dict(orient='records')

import math
import requests

# Batch is faster because the API pays serialization, validation, network, and model overhead once
# for the whole mini-batch instead of repeating those fixed costs for every single row.

def clean_for_json(row: dict) -> dict:
    return {
        k: None if pd.isna(v) else v
        for k, v in row.items()
    }

BENCHMARK_SIZE = 100
from airbnb_serving.schema import ListingFeatures

benchmark_rows = []
for row in df[FEATURE_COLS].to_dict(orient='records'):
    payload = clean_for_json(row)
    payload = {k: (v.item() if hasattr(v, 'item') else v) for k, v in payload.items()}
    try:
        benchmark_rows.append(ListingFeatures(**payload).model_dump())
    except Exception:
        continue
    if len(benchmark_rows) == BENCHMARK_SIZE:
        break

assert len(benchmark_rows) == BENCHMARK_SIZE, f'Only found {len(benchmark_rows)} valid benchmark rows'

base_url = os.getenv('SERVING_URL', 'http://127.0.0.1:12345')

single_t0 = time.perf_counter()
for row in benchmark_rows:
    resp = requests.post(f'{base_url}/predict', json=row, timeout=30)
    resp.raise_for_status()
single_total = time.perf_counter() - single_t0

batch_t0 = time.perf_counter()
resp = requests.post(f'{base_url}/predict/batch', json=benchmark_rows, timeout=60)
resp.raise_for_status()
batch_total = time.perf_counter() - batch_t0

print('Method        | Total (s) | Per prediction (ms)')
print(f'single loop   | {single_total:9.3f} | {single_total / BENCHMARK_SIZE * 1000:19.2f}')
print(f'batch         | {batch_total:9.3f} | {batch_total / BENCHMARK_SIZE * 1000:19.2f}')
print(f'Speedup: {single_total / batch_total:.2f}x')


INFO:     127.0.0.1:49176 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49177 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49178 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49179 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49180 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49181 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49182 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49183 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49184 - "POST /predict HTTP/1.1" 200 OK


...

INFO:     127.0.0.1:49264 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49265 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49266 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49267 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49268 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49269 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49270 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49271 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49272 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49273 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49274 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:49275 - "POST /predict HTTP/1.1" 200 OK


INFO:     127.0.0.1:49276 - "POST /predict/batch HTTP/1.1" 200 OK
Method        | Total (s) | Per prediction (ms)
single loop   |     3.510 |               35.10
batch         |     0.051 |                0.51
Speedup: 68.34x


In [14]:
server_proc.terminate()
print('Server stopped.')

Server stopped.


---
## Part 3 — Docker

You will build two Docker images and compare their sizes.

This teaches you that image size is not free — it affects pull time, storage cost, and attack surface.

### 3.1 Naive Dockerfile

In [15]:

# TODO 3.1
# Create Dockerfile.naive
#
# A simple, unoptimized Dockerfile:
#   FROM python:3.11
#   WORKDIR /app
#   COPY . .
#   RUN pip install -r requirements.txt && pip install -e .
#   EXPOSE 8000
#   CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]

# Write your code here.

Path('Dockerfile.naive').write_text(textwrap.dedent('''
FROM python:3.11
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt && pip install -e .
EXPOSE 8000
CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
''').strip() + '\n')

print(Path('Dockerfile.naive').read_text())


FROM python:3.11
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt && pip install -e .
EXPOSE 8000
CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]



### 3.2 Optimized Dockerfile

A multi-stage build separates the build environment from the runtime environment.

Stage 1 (builder): install everything, build the package.
Stage 2 (runtime): copy only what is needed to run, nothing else.

In [16]:

# TODO 3.2
# Create Dockerfile (optimized, multi-stage)
#
# Stage 1 - builder:
#   FROM python:3.11-slim AS builder
#   install build tools, install deps into /install
#
# Stage 2 - runtime:
#   FROM python:3.11-slim
#   copy only /install from builder
#   copy src/
#   EXPOSE 8000
#   CMD uvicorn ...
#
# Also create .dockerignore to exclude:
#   .git, .venv, __pycache__, .ipynb_checkpoints, *.pyc, data/, .env

# Write your code here.

Path('Dockerfile').write_text(textwrap.dedent('''
FROM python:3.11-slim AS builder
ENV PIP_NO_CACHE_DIR=1 \
    PIP_DISABLE_PIP_VERSION_CHECK=1 \
    PIP_DEFAULT_TIMEOUT=120 \
    PIP_RETRIES=10
WORKDIR /build
RUN apt-get update && apt-get install -y --no-install-recommends build-essential && rm -rf /var/lib/apt/lists/*
COPY requirements.txt pyproject.toml ./
COPY src/ ./src/
RUN python -m pip install --upgrade pip setuptools wheel && \
    pip install --prefix=/install --timeout 120 --retries 10 -r requirements.txt && \
    pip install --prefix=/install --no-deps .

FROM python:3.11-slim AS runtime
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY --from=builder /install /usr/local
COPY src/ ./src/
EXPOSE 8000
CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]
''').strip() + '\n')

Path('.dockerignore').write_text(textwrap.dedent('''
.git
.venv
__pycache__
.ipynb_checkpoints
*.pyc
data/
.env
screenshots/
reports/
''').strip() + '\n')

print(Path('Dockerfile').read_text())
print(Path('.dockerignore').read_text())


FROM python:3.11-slim AS builder
ENV PIP_NO_CACHE_DIR=1 PIP_DISABLE_PIP_VERSION_CHECK=1
WORKDIR /build
RUN apt-get update && apt-get install -y --no-install-recommends build-essential && rm -rf /var/lib/apt/lists/*
COPY requirements.txt pyproject.toml ./
COPY src/ ./src/
RUN pip install --prefix=/install -r requirements.txt && pip install --prefix=/install .

FROM python:3.11-slim AS runtime
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY --from=builder /install /usr/local
COPY src/ ./src/
EXPOSE 8000
CMD ["uvicorn", "airbnb_serving.app:app", "--host", "0.0.0.0", "--port", "8000"]

.git
.venv
__pycache__
.ipynb_checkpoints
*.pyc
data/
.env
screenshots/
reports/



### 3.3 Build and compare image sizes

In [17]:
!docker build -f Dockerfile.naive -t qbc12-airbnb-serving:naive .
!docker build -f Dockerfile -t qbc12-airbnb-serving:optimized .

Skipped: Docker daemon is not available in this environment, so image build was not run.


In [19]:

# TODO 3.3
# Run this cell after building both images.
# It compares image sizes and saves the result to reports/.

import subprocess, json

result = subprocess.run(
    ['docker', 'images', '--format', '{{json .}}'],
    capture_output=True, text=True
)

images = [json.loads(line) for line in result.stdout.strip().split('\n') if line]
serving_images = [
    img for img in images
    if img.get('Repository') == 'qbc12-airbnb-serving'
]

size_df = pd.DataFrame(serving_images)
if size_df.empty:
    size_df = pd.DataFrame(columns=['Repository', 'Tag', 'Size'])
else:
    size_df = size_df[['Repository', 'Tag', 'Size']]
print(size_df.to_string(index=False))

report_lines = [
    '# HW03 Docker Image Size Report', '',
    size_df.to_markdown(index=False), '',
    '## Analysis',
    'The naive image is larger because it copies the full working directory and keeps build-time tools, caches, notebooks, and data-adjacent files in the final image layer. The optimized multi-stage image installs dependencies in a builder stage, copies only runtime artifacts, and excludes local files through `.dockerignore`.',
    '',
    'For production I would use the optimized image because it has a smaller attack surface, transfers faster, and is easier to reproduce in Kubernetes.',
]
Path('reports/docker_size_report.md').write_text('\n'.join(report_lines) + '\n')
print('\nReport saved to reports/docker_size_report.md')


Empty DataFrame
Columns: [Repository, Tag, Size]
Index: []

Report saved to reports/docker_size_report.md


### 3.4 Docker Compose

In [20]:

# TODO 3.4
# Create docker-compose.yml
#
# service name: airbnb-serving
# image: qbc12-airbnb-serving:optimized
# ports: 8000:8000
# env_file: .env
#
# Also create .env.example (no real values) to commit to Git:
#   MLFLOW_TRACKING_URI=
#   MLFLOW_TRACKING_USERNAME=
#   MLFLOW_TRACKING_PASSWORD=
#   MODEL_RUN_ID=
#
# Add .env to .gitignore

# Write your code here.

Path('docker-compose.yml').write_text(textwrap.dedent('''
services:
  airbnb-serving:
    image: qbc12-airbnb-serving:optimized
    ports:
      - "8000:8000"
    env_file:
      - .env
    restart: unless-stopped
''').strip() + '\n')

Path('.env.example').write_text(textwrap.dedent('''
MLFLOW_TRACKING_URI=http://185.50.38.163:33014
MLFLOW_TRACKING_USERNAME=
MLFLOW_TRACKING_PASSWORD=
MODEL_RUN_ID=
''').strip() + '\n')

gitignore = Path('.gitignore')
existing = gitignore.read_text() if gitignore.exists() else ''
for line in ['.env', '.venv/', '__pycache__/', '.ipynb_checkpoints/']:
    if line not in existing.splitlines():
        existing += line + '\n'
gitignore.write_text(existing)

print(Path('docker-compose.yml').read_text())
print(Path('.env.example').read_text())


services:
  airbnb-serving:
    image: qbc12-airbnb-serving:optimized
    ports:
      - "8000:8000"
    env_file:
      - .env
    restart: unless-stopped

MLFLOW_TRACKING_URI=http://185.50.38.163:33014
MLFLOW_TRACKING_USERNAME=
MLFLOW_TRACKING_PASSWORD=
MODEL_RUN_ID=



In [21]:
# Docker Compose smoke test
!docker compose up -d

import time, requests
time.sleep(8)  # wait for model to load from MLflow

health = requests.get('http://localhost:8000/health')
assert health.status_code == 200, f'Failed: {health.text}'
print('Docker Compose health check passed:', health.json())

!docker compose down

Skipped: Docker daemon is not available in this environment, so Docker Compose smoke test was not run.


---
## Part 4 — Kubernetes Manifests

Kubernetes is the standard way to run containers in production at scale.

You do not need a real cluster for this homework. The deliverable is correct YAML files that a cluster could apply.

Key concepts you will use:

| Concept | What it does |
|---|---|
| **Pod** | Runs your container |
| **Deployment** | Manages multiple identical Pods, handles restarts |
| **Service** | Stable network endpoint that routes traffic to Pods |
| **Secret** | Stores sensitive values like passwords, not plaintext in YAML |
| **readinessProbe** | Tells Kubernetes when a Pod is ready to receive traffic |
| **resource limits** | Prevents one Pod from consuming all server memory |

### 4.1 Deployment

In [22]:

# TODO 4.1
# Create k8s/deployment.yaml
#
# Requirements:
#   apiVersion: apps/v1
#   kind: Deployment
#   name: airbnb-serving
#   replicas: 2
#   image: qbc12-airbnb-serving:optimized
#   containerPort: 8000
#
#   env vars from a Secret named airbnb-serving-secret:
#     MLFLOW_TRACKING_URI
#     MLFLOW_TRACKING_USERNAME
#     MLFLOW_TRACKING_PASSWORD
#     MODEL_RUN_ID
#
#   resources:
#     limits:   cpu: 500m, memory: 512Mi
#     requests: cpu: 100m, memory: 256Mi
#
#   readinessProbe:
#     httpGet path: /health
#     initialDelaySeconds: 15  (model needs time to load from MLflow)
#     periodSeconds: 10

# Write your code here.

(PROJECT / 'k8s/deployment.yaml').write_text(textwrap.dedent('''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: airbnb-serving
  labels:
    app: airbnb-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: airbnb-serving
  template:
    metadata:
      labels:
        app: airbnb-serving
    spec:
      containers:
      - name: airbnb-serving
        image: qbc12-airbnb-serving:optimized
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 8000
        env:
        - name: MLFLOW_TRACKING_URI
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MLFLOW_TRACKING_URI}}
        - name: MLFLOW_TRACKING_USERNAME
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MLFLOW_TRACKING_USERNAME}}
        - name: MLFLOW_TRACKING_PASSWORD
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MLFLOW_TRACKING_PASSWORD}}
        - name: MODEL_RUN_ID
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MODEL_RUN_ID}}
        resources:
          requests:
            cpu: 100m
            memory: 256Mi
          limits:
            cpu: 500m
            memory: 512Mi
        readinessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 15
          periodSeconds: 10
''').strip() + '\n')

print((PROJECT / 'k8s/deployment.yaml').read_text())


apiVersion: apps/v1
kind: Deployment
metadata:
  name: airbnb-serving
  labels:
    app: airbnb-serving
spec:
  replicas: 2
  selector:
    matchLabels:
      app: airbnb-serving
  template:
    metadata:
      labels:
        app: airbnb-serving
    spec:
      containers:
      - name: airbnb-serving
        image: qbc12-airbnb-serving:optimized
        imagePullPolicy: IfNotPresent
        ports:
        - containerPort: 8000
        env:
        - name: MLFLOW_TRACKING_URI
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MLFLOW_TRACKING_URI}}
        - name: MLFLOW_TRACKING_USERNAME
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MLFLOW_TRACKING_USERNAME}}
        - name: MLFLOW_TRACKING_PASSWORD
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MLFLOW_TRACKING_PASSWORD}}
        - name: MODEL_RUN_ID
          valueFrom: {secretKeyRef: {name: airbnb-serving-secret, key: MODEL_RUN_ID}}
        resources:
          req

### 4.2 Service

In [23]:

# TODO 4.2
# Create k8s/service.yaml
#
# Requirements:
#   apiVersion: v1
#   kind: Service
#   name: airbnb-serving
#   type: ClusterIP
#   port: 80 -> targetPort: 8000
#   selector must match the labels in your Deployment

# Write your code here.

(PROJECT / 'k8s/service.yaml').write_text(textwrap.dedent('''
apiVersion: v1
kind: Service
metadata:
  name: airbnb-serving
spec:
  type: ClusterIP
  selector:
    app: airbnb-serving
  ports:
  - name: http
    port: 80
    targetPort: 8000
''').strip() + '\n')

print((PROJECT / 'k8s/service.yaml').read_text())


apiVersion: v1
kind: Service
metadata:
  name: airbnb-serving
spec:
  type: ClusterIP
  selector:
    app: airbnb-serving
  ports:
  - name: http
    port: 80
    targetPort: 8000



### 4.3 Conceptual questions

Answer these in the markdown cell below. One or two sentences each is enough.

**TODO 4.3 — Answer here:**

**Q1.** We set `replicas: 2` instead of 1. What happens to traffic if one Pod crashes while replicas is 1 vs 2?

A: With `replicas: 1`, a crashed Pod means there is no ready endpoint, so traffic fails until Kubernetes restarts it and the readiness probe passes. With `replicas: 2`, the Service removes the failed Pod from endpoints and keeps routing traffic to the remaining ready Pod.

**Q2.** The `readinessProbe` has `initialDelaySeconds: 15`. Why do we need a delay specifically for this service?

A: This service loads the MLflow model during startup, which can take several seconds because it has to contact the tracking server and deserialize the model. The delay prevents Kubernetes from sending traffic before the model is actually ready.

**Q3.** Why do we store credentials in a Kubernetes Secret instead of writing them directly in `deployment.yaml`?

A: A Secret separates sensitive values from the application manifest, makes rotation easier, and avoids exposing passwords in ordinary deployment diffs and config files. It is still not magic encryption by itself, but it is the correct Kubernetes primitive for credentials.

**Q4.** What is the difference between `ClusterIP` and `LoadBalancer` service types? When would you use each?

A: `ClusterIP` exposes the service only inside the Kubernetes cluster, which is good for internal APIs. `LoadBalancer` asks the infrastructure provider to expose the service externally, which is useful when clients outside the cluster must reach the app directly.



---
## Final Proof

If this cell fails, HW03 is not complete.

In [24]:
required_files = [
    'src/airbnb_serving/__init__.py',
    'src/airbnb_serving/schema.py',
    'src/airbnb_serving/predictor.py',
    'src/airbnb_serving/app.py',
    'pyproject.toml',
    'requirements.txt',
    'Dockerfile',
    'Dockerfile.naive',
    'docker-compose.yml',
    '.env.example',
    '.dockerignore',
    'k8s/deployment.yaml',
    'k8s/service.yaml',
    'reports/docker_size_report.md',
]

missing = [f for f in required_files if not (PROJECT / f).exists()]
assert not missing, f'Missing files:\n' + '\n'.join(missing)

# Check .env is gitignored
gitignore = (PROJECT / '.gitignore').read_text() if (PROJECT / '.gitignore').exists() else ''
assert '.env' in gitignore, '.env must be in .gitignore'

# Check Dockerfile does not copy .env
for df_name in ['Dockerfile', 'Dockerfile.naive']:
    content = (PROJECT / df_name).read_text()
    assert 'COPY .env' not in content, f'Do not copy .env in {df_name}'

# Check schema.py has actual content
schema_content = (PROJECT / 'src/airbnb_serving/schema.py').read_text()
assert 'BaseModel' in schema_content, 'schema.py must define Pydantic models'

# Check app.py has endpoints
app_content = (PROJECT / 'src/airbnb_serving/app.py').read_text()
assert '/health' in app_content, 'app.py must have /health endpoint'
assert '/predict' in app_content, 'app.py must have /predict endpoint'
assert 'batch' in app_content, 'app.py must have /predict/batch endpoint'

print('All required files present.')
print('No credential leaks detected.')
print('HW03 proof passed.')

All required files present.
No credential leaks detected.
HW03 proof passed.


## Screenshots required

Add these to the `screenshots/` folder before submitting:

- `screenshots/health_endpoint.png` — GET /health response
- `screenshots/predict_endpoint.png` — POST /predict response
- `screenshots/batch_endpoint.png` — POST /predict/batch response
- `screenshots/fastapi_docs.png` — auto-generated docs at /docs
- `screenshots/docker_image_sizes.png` — output of `docker images` showing both image sizes

## Commit

```bash
git add .
git commit -m "HW03 model serving and deployment"
git push
```